# Experimento C: Validación de Integridad Metodológica (The Anti-Leakage Test)

## Objetivo
Demostrar empíricamente cómo la literatura falla al aplicar transformaciones antes de la división de datos.  
Este experimento **valida la contribución metodológica** del TFM.

## Dos Ramas
- **Rama "Incorrecta" (Simulación de Leakage)**: Aplicar transformaciones (SMOTE / escalado global) a todo el dataset antes de dividir en Train/Test.
- **Rama "Correcta" (Nuestra Propuesta)**: Dividir primero en Train/Test cronológicamente. Aplicar transformaciones solo en Train y proyectar en Test.

## Resultado Esperado
- La rama "Incorrecta" dará resultados sospechosamente altos (ej. AUPRC > 0.95)
- La "Correcta" dará resultados realistas (ej. AUPRC ~0.80)

## Valor
Este gráfico será la **evidencia visual** de la crítica al "Estado del Arte" deficiente.

In [ ]:
import os, sys, warnings, pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn import metrics
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))

from experiments.config import (
    SEED, INPUT_FEATURES, OUTPUT_FEATURE,
    RESULTS_DIR, FIGURES_DIR, COLORS,
    START_DATE_TRAINING, DELTA_TRAIN, DELTA_DELAY, DELTA_TEST,
)
from experiments.data_utils import (
    load_transformed_data, get_train_test_set,
    print_dataset_summary,
)

warnings.filterwarnings('ignore')
sns.set_style('darkgrid', {'axes.facecolor': '0.9'})
print("Configuración del Experimento C cargada")

---
## 1. Carga de Datos

In [ ]:
transactions_df = load_transformed_data()
print(f"Dataset total: {len(transactions_df):,} transacciones")

---
## 2. Rama CORRECTA: Split temporal primero, luego transformar

In [ ]:
# Rama CORRECTA: División temporal estricta -> SMOTE solo en train -> Evaluar en test
train_df, test_df = get_train_test_set(
    transactions_df,
    start_date_training=START_DATE_TRAINING,
    delta_train=DELTA_TRAIN, delta_delay=DELTA_DELAY, delta_test=DELTA_TEST,
)
print_dataset_summary(train_df, test_df, "Rama CORRECTA (sin leakage)")

# Pipeline correcto: SMOTE solo se aplica en train
correct_pipeline = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=SEED)),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)),
])

# Entrenar solo con train, evaluar en test
correct_pipeline.fit(train_df[INPUT_FEATURES], train_df[OUTPUT_FEATURE])

y_pred_correct = correct_pipeline.predict_proba(test_df[INPUT_FEATURES])[:, 1]

auprc_correct = metrics.average_precision_score(test_df[OUTPUT_FEATURE], y_pred_correct)
auc_correct = metrics.roc_auc_score(test_df[OUTPUT_FEATURE], y_pred_correct)

print(f"\n  Rama CORRECTA:")
print(f"    AUC ROC: {auc_correct:.4f}")
print(f"    AUPRC:   {auprc_correct:.4f}  ← resultado REALISTA")

---
## 3. Rama INCORRECTA: Transformar todo el dataset antes de dividir (Data Leakage)

In [ ]:
# Rama INCORRECTA: Aplicar SMOTE a TODO el dataset, luego dividir
# (Esto es lo que mucha literatura hace mal)

# 1. Escalar todo el dataset (leakage: el scaler ve los datos de test)
scaler_global = StandardScaler()
X_all_scaled = scaler_global.fit_transform(transactions_df[INPUT_FEATURES])
y_all = transactions_df[OUTPUT_FEATURE].values

# 2. Aplicar SMOTE a todo (leakage: genera muestras sintéticas con info de test)
smote = SMOTE(random_state=SEED)
X_resampled, y_resampled = smote.fit_resample(X_all_scaled, y_all)
print(f"Después de SMOTE global: {len(X_resampled):,} muestras (originales: {len(X_all_scaled):,})")

# 3. Dividir DESPUÉS de SMOTE (train/test split simple, no temporal)
from sklearn.model_selection import train_test_split
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_resampled, y_resampled, test_size=0.3, random_state=SEED, stratify=y_resampled
)

# 4. Entrenar y evaluar
clf_incorrect = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
clf_incorrect.fit(X_train_leak, y_train_leak)

y_pred_incorrect = clf_incorrect.predict_proba(X_test_leak)[:, 1]

auprc_incorrect = metrics.average_precision_score(y_test_leak, y_pred_incorrect)
auc_incorrect = metrics.roc_auc_score(y_test_leak, y_pred_incorrect)

print(f"\n  Rama INCORRECTA (con Data Leakage):")
print(f"    AUC ROC: {auc_incorrect:.4f}")
print(f"    AUPRC:   {auprc_incorrect:.4f}  ← resultado INFLADO artificialmente")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras: AUPRC comparación
ax = axes[0]
methods = ['Correcta\n(Sin Leakage)', 'Incorrecta\n(Con Leakage)']
auprcs = [auprc_correct, auprc_incorrect]
colors_bars = [COLORS['correct_pipeline'], COLORS['incorrect_pipeline']]
bars = ax.bar(methods, auprcs, color=colors_bars, edgecolor='black', width=0.5)
for bar, val in zip(bars, auprcs):
    ax.annotate(f'{val:.4f}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                ha='center', va='bottom', fontsize=14, fontweight='bold')
ax.set_ylabel('AUPRC', fontsize=13)
ax.set_title('Impacto del Data Leakage en AUPRC\n(Random Forest + SMOTE)', fontsize=14)
ax.set_ylim([0, 1.1])

# Curvas PR
ax = axes[1]
prec_c, rec_c, _ = metrics.precision_recall_curve(test_df[OUTPUT_FEATURE], y_pred_correct)
ax.plot(rec_c, prec_c, color=COLORS['correct_pipeline'], linewidth=2,
        label=f'Correcta (AP={auprc_correct:.3f})')
prec_i, rec_i, _ = metrics.precision_recall_curve(y_test_leak, y_pred_incorrect)
ax.plot(rec_i, prec_i, color=COLORS['incorrect_pipeline'], linewidth=2, linestyle='--',
        label=f'Incorrecta (AP={auprc_incorrect:.3f})')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Curvas PR: Pipeline Correcto vs Incorrecto', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim([0, 1.01]); ax.set_ylim([0, 1.01])

plt.tight_layout()
fig.savefig(FIGURES_DIR / 'experiment_c_leakage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Guardar resultados
results_c = {
    'correct': {'auc_roc': auc_correct, 'auprc': auprc_correct},
    'incorrect': {'auc_roc': auc_incorrect, 'auprc': auprc_incorrect},
}
with open(RESULTS_DIR / 'experiment_c_results.pkl', 'wb') as f:
    pickle.dump(results_c, f)

print("✓ Resultados del Experimento C guardados")

---
## 5. Tabla Comparativa A (Realista) vs C-Incorrecto (Inflado)

In [ ]:
comparison_c = pd.DataFrame({
    'Pipeline': ['Correcta (Sin Leakage)', 'Incorrecta (Con Leakage)'],
    'AUC ROC': [auc_correct, auc_incorrect],
    'AUPRC': [auprc_correct, auprc_incorrect],
    'Diferencia AUPRC': ['-', f'+{(auprc_incorrect - auprc_correct):.4f} (inflado)'],
}).set_index('Pipeline')

print("\nTabla comparativa - Experimento C:")
print("=" * 80)
display(comparison_c.round(4))

comparison_c.to_csv(RESULTS_DIR / 'experiment_c_comparison.csv')
print("\n✓ Tabla guardada")

---
## 6. Conclusiones del Experimento C

**Este gráfico es la evidencia visual de la crítica al Estado del Arte deficiente.**

La diferencia entre ambas ramas demuestra cómo el Data Leakage infla artificialmente los resultados, dando una falsa sensación de seguridad sobre el rendimiento del modelo.